# Research notebook archive

Original project notebook with outputs and personal metadata removed. Existing Colab paths and experiment settings require configuration; see the repository README before running.

# Feature ablation comparison: base vs demographic fine-tuned model

This Colab notebook compares how the base Qwen2.5-14B model and your demographic LoRA fine-tuned model use demographic/context features in the Stage 1 demographic-inference task.

It follows the logic of your old feature-ablation notebook, but makes the comparison paired and model-comparative:

- uses the same sampled rows for both models
- uses the same answer-letter maps for both models
- re-scores both the baseline full-profile prompt and every leave-one-feature-out prompt for both models
- saves all letter log probabilities, class log probabilities, class probabilities, prompts, and option maps
- outputs base-vs-fine-tuned plots and tables


In [ ]:
# Colab install cell. Restart runtime after first install if Colab asks.
!pip install -q transformers peft accelerate pyarrow pandas numpy tqdm matplotlib requests
# PEFT can crash if Colab has an old optional torchao installed. We do not need torchao here.
!pip uninstall -y torchao


In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')


In [ ]:
# =====================================================================
# USER CONFIG: change only this block first
# =====================================================================
from pathlib import Path
import json

PROJECT_ROOT = Path('/content/drive/MyDrive/NLP_project')
DATA_DIR = PROJECT_ROOT / 'data'
OUT_ROOT = DATA_DIR / 'feature_ablation_compare'

# Same base + LoRA adapter setup as your behavioral spillover notebook.
BASE_MODEL_PATH = 'Qwen/Qwen2.5-14B-Instruct'
FT_MODEL_PATH = (
    '/content/drive/MyDrive/NLP_project/data/stage3_outputs/'
    'sequential_experiments/balanced_canonical/checkpoints/step_07_after_ENGLISH_STATUS'
)
BASE_RUN_NAME = 'qwen25_14b_base'
FT_RUN_NAME = 'qwen25_14b_demog_lora_step07'

# Sampling frame: folder with phase1_scored_RACE_ETHNICITY.parquet, etc.
# Update this if the resolver cannot find your Stage 1 base-model output folder.
STAGE1_FRAME_CANDIDATES = [
    DATA_DIR / 'stage_1_output_Qwen2.5-14B-Instruct_baseline_test_balanced_05_11',
    DATA_DIR / 'stage_1_output_Qwen2.5-14B-Instruct_baseline',
    DATA_DIR / 'stage_1_output_Qwen/Qwen2.5-14B-Instruct',
    DATA_DIR / 'stage_1_output_Qwen3-4B-Instruct-2507',
]

ABLATION_TARGETS = ['RACE_ETHNICITY', 'SEX', 'AGE_BINS']
FOCUS_GROUPS = {
    'RACE_ETHNICITY': [['White']],
    'SEX': [['Female']],
    'AGE_BINS': [['35-44', '45-54']],  # normalized later to match dash style if needed
}

# Smoke test: 20-50. Real run: 100-200. Larger multiplies quickly.
N_ABLATION_PER_CLASS = 100
RANDOM_SEED = 42
BATCH_SIZE = 4       # lower to 1-2 if OOM, raise to 8 if memory is fine
MAX_MODEL_LEN = 4096
PRIMING_SUFFIX = ''
USE_INFERENCE_ROWS_ONLY = True

# Use raw probabilities for fair base-vs-FT comparison.
# The old notebook used Stage 1 PriDe letter priors, but those priors are model-specific.
USE_STAGE1_LETTER_PRIOR = False

RUN_TAG = f'ablation_compare_n{N_ABLATION_PER_CLASS}_rawprobs_targets_{"-".join(ABLATION_TARGETS)}'
RUN_ID = f'{BASE_RUN_NAME}__vs__{FT_RUN_NAME}__seed{RANDOM_SEED}__{RUN_TAG}'
OUT_DIR = OUT_ROOT / RUN_ID
SCORES_DIR = OUT_DIR / 'scores'
TABLES_DIR = OUT_DIR / 'tables'
PLOTS_DIR = OUT_DIR / 'plots'
PROGRESS_DIR = OUT_DIR / 'progress'
for p in [OUT_ROOT, OUT_DIR, SCORES_DIR, TABLES_DIR, PLOTS_DIR, PROGRESS_DIR]:
    p.mkdir(parents=True, exist_ok=True)


def resolve_stage1_frame_dir(candidates, targets):
    print('Checking Stage 1 sampling-frame candidates:')
    for p in candidates:
        ok = p.exists() and all((p / f'phase1_scored_{t}.parquet').exists() for t in targets)
        print(' ', p, 'exists=', p.exists(), 'has_all_targets=', ok)
        if ok:
            return p
    raise FileNotFoundError(
        'Could not find a Stage 1 folder with phase1_scored_<TARGET>.parquet for all targets.\n'
        'Update STAGE1_FRAME_CANDIDATES in this cell.'
    )

STAGE1_FRAME_DIR = resolve_stage1_frame_dir(STAGE1_FRAME_CANDIDATES, ABLATION_TARGETS)

print('\nPROJECT_ROOT:', PROJECT_ROOT)
print('STAGE1_FRAME_DIR:', STAGE1_FRAME_DIR)
print('OUT_DIR:', OUT_DIR)
print('BASE_MODEL_PATH:', BASE_MODEL_PATH)
print('FT_MODEL_PATH:', FT_MODEL_PATH, 'exists=', Path(FT_MODEL_PATH).exists())
print('N_ABLATION_PER_CLASS:', N_ABLATION_PER_CLASS)
print('ABLATION_TARGETS:', ABLATION_TARGETS)

ft_path = Path(FT_MODEL_PATH)
if not ft_path.exists():
    raise FileNotFoundError(f'FT_MODEL_PATH does not exist: {FT_MODEL_PATH}')
is_lora_adapter = (ft_path / 'adapter_config.json').exists() and (
    (ft_path / 'adapter_model.safetensors').exists() or (ft_path / 'adapter_model.bin').exists()
)
if not is_lora_adapter:
    raise FileNotFoundError(f'Expected FT_MODEL_PATH to be a LoRA adapter folder: {FT_MODEL_PATH}')
adapter_cfg = json.loads((ft_path / 'adapter_config.json').read_text())
print('Adapter base_model_name_or_path:', adapter_cfg.get('base_model_name_or_path'))
if adapter_cfg.get('base_model_name_or_path') != BASE_MODEL_PATH:
    raise ValueError('Adapter base_model_name_or_path does not match BASE_MODEL_PATH.')


In [ ]:
# =====================================================================
# Imports and general helpers
# =====================================================================
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import gc
import json
import math
import re
import string
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 120)
pd.set_option('display.precision', 4)
plt.rcParams['figure.dpi'] = 120


def cleanup_memory():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass


def safe_name(x: str) -> str:
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(x)).strip('_')


def softmax(logits):
    x = np.asarray(logits, dtype=np.float64)
    x = x - np.nanmax(x)
    e = np.exp(x)
    return e / e.sum()


def weighted_mean(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    mask = np.isfinite(x) & np.isfinite(w) & (w > 0)
    if mask.sum() == 0:
        return np.nan
    return float(np.sum(x[mask] * w[mask]) / np.sum(w[mask]))


def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        json.dump(obj, f, indent=2, default=str)

save_json({
    'BASE_MODEL_PATH': BASE_MODEL_PATH,
    'FT_MODEL_PATH': FT_MODEL_PATH,
    'BASE_RUN_NAME': BASE_RUN_NAME,
    'FT_RUN_NAME': FT_RUN_NAME,
    'STAGE1_FRAME_DIR': str(STAGE1_FRAME_DIR),
    'ABLATION_TARGETS': ABLATION_TARGETS,
    'FOCUS_GROUPS': FOCUS_GROUPS,
    'N_ABLATION_PER_CLASS': N_ABLATION_PER_CLASS,
    'RANDOM_SEED': RANDOM_SEED,
    'BATCH_SIZE': BATCH_SIZE,
    'MAX_MODEL_LEN': MAX_MODEL_LEN,
    'USE_STAGE1_LETTER_PRIOR': USE_STAGE1_LETTER_PRIOR,
    'USE_INFERENCE_ROWS_ONLY': USE_INFERENCE_ROWS_ONLY,
}, OUT_DIR / 'run_config.json')


In [ ]:
# =====================================================================
# Prompt metadata copied/adapted from Stage 1 and the old ablation notebook
# =====================================================================
FEATURE_LABELS = {
    'SEX': 'Sex',
    'AGE_BINS': 'Age group',
    'RACE_ETHNICITY': 'Race/ethnicity',
    'EDUCATION_LEVEL': 'Highest level of education completed',
    'INCTOT_BINS': 'Total annual personal income',
    'HEALTH_INSURANCE': 'Health insurance coverage',
    'ENGLISH_STATUS': 'English speaking ability',
    'NCHILD': 'Number of own children living in household',
    'UHRSWORK': 'Usual hours worked per week',
    'STATE_NAME': 'State of residence',
    'OWNERSHP_STATUS': 'Home ownership status',
    'MARITAL_STATUS': 'Marital status',
    'EMPLOYMENT_STATUS': 'Employment status',
    'WORKER_CLASS': 'Class of worker',
    'TRANSPORT_METHOD': 'Means of transportation to work',
}

LABEL_TO_LONG = {
    'White': 'White',
    'Hispanic': 'Hispanic',
    'Black': 'Black or African American',
    'Indigenous': 'American Indian or Alaska Native',
    'Chinese': 'Chinese',
    'Japanese': 'Japanese',
    'Other': 'Other Asian or Pacific Islander',
    '18-24': '18 to 24 years old',
    '25-34': '25 to 34 years old',
    '35-44': '35 to 44 years old',
    '45-54': '45 to 54 years old',
    '55-64': '55 to 64 years old',
    '18–24': '18 to 24 years old',
    '25–34': '25 to 34 years old',
    '35–44': '35 to 44 years old',
    '45–54': '45 to 54 years old',
    '55–64': '55 to 64 years old',
    '65+': '65 years or older',
    'Male': 'Male',
    'Female': 'Female',
}

SYSTEM_PROMPT = '''You are completing one missing demographic attribute for a U.S. survey respondent.

You will be given a profile with one missing attribute and a list of lettered answer options. Predict the most likely value of that missing attribute using the other provided attributes.

Rules:
- Use only the provided profile.
- Choose exactly one option from the list.
- Reply with only the letter of the selected option -- no parentheses, no explanation, no extra text.'''


def normalize_label(x):
    if isinstance(x, str):
        return x.replace('–', '-')
    return x


def _display_label(class_label: str) -> str:
    return LABEL_TO_LONG.get(class_label, LABEL_TO_LONG.get(normalize_label(class_label), str(class_label)))


def build_profile_lines(row: pd.Series, target_col: str, drop_features=()) -> list[str]:
    excluded = set(drop_features) | {target_col}
    lines = []
    for col, label in FEATURE_LABELS.items():
        if col in excluded or col not in row.index:
            continue
        value = row[col]
        if pd.isna(value):
            continue
        display_value = LABEL_TO_LONG.get(value, LABEL_TO_LONG.get(normalize_label(value), value))
        lines.append(f'- {label}: {display_value}')
    return lines


def build_user_prompt(row: pd.Series, target_col: str, lettered_options: list[tuple[str, str]], drop_features=()) -> str:
    profile_text = '\n'.join(build_profile_lines(row, target_col, drop_features=drop_features))
    answers_text = '\n'.join(f'({letter}) {label}' for letter, label in lettered_options)
    target_label = FEATURE_LABELS[target_col]
    return (
        f'Profile:\n\n{profile_text}\n\n'
        f'Missing attribute: {target_label}\n\n'
        f'Answer options:\n{answers_text}\n\n'
        f'Reply with only the letter of the correct option.'
    )


def context_features_for(scored: pd.DataFrame, target_col: str) -> list[str]:
    return [c for c in FEATURE_LABELS if c != target_col and c in scored.columns]


In [ ]:
# =====================================================================
# Transformers scorer with LoRA/PEFT adapter support
# =====================================================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

class LetterScorer:
    def __init__(self, model_name_or_path: str, model_label: str):
        cleanup_memory()
        self.model_name_or_path = model_name_or_path
        self.model_label = model_label
        model_path = Path(model_name_or_path)
        self.is_lora_adapter = model_path.exists() and (model_path / 'adapter_config.json').exists()
        dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

        if self.is_lora_adapter and ((model_path / 'tokenizer.json').exists() or (model_path / 'tokenizer_config.json').exists()):
            tokenizer_source = model_name_or_path
        elif self.is_lora_adapter:
            tokenizer_source = BASE_MODEL_PATH
        else:
            tokenizer_source = model_name_or_path

        print(f'Initializing tokenizer from: {tokenizer_source}')
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_source, trust_remote_code=False)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = 'left'

        print(f'Initializing model for {model_label}: {model_name_or_path}')
        print('  dtype:', dtype)
        print('  device_map: auto')
        if self.is_lora_adapter:
            from peft import PeftModel
            adapter_cfg = json.loads((model_path / 'adapter_config.json').read_text())
            adapter_base = adapter_cfg.get('base_model_name_or_path', BASE_MODEL_PATH)
            print('Detected LoRA/PEFT adapter')
            print('  base model:', adapter_base)
            print('  adapter:', model_name_or_path)
            base_model = AutoModelForCausalLM.from_pretrained(
                adapter_base,
                torch_dtype=dtype,
                device_map='auto',
                trust_remote_code=False,
                low_cpu_mem_usage=True,
            )
            self.model = PeftModel.from_pretrained(
                base_model,
                model_name_or_path,
                torch_dtype=dtype,
                device_map='auto',
            )
        else:
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name_or_path,
                torch_dtype=dtype,
                device_map='auto',
                trust_remote_code=False,
                low_cpu_mem_usage=True,
            )
        self.model.eval()
        self._letter_token_cache = {}

    def close(self):
        try:
            del self.model
        except Exception:
            pass
        cleanup_memory()

    def _build_prefix(self, system_prompt: str, user_prompt: str) -> str:
        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ]
        return self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def _single_token_id(self, letter: str) -> int:
        if letter in self._letter_token_cache:
            return self._letter_token_cache[letter]
        ids = self.tokenizer(letter, add_special_tokens=False).input_ids
        if len(ids) != 1:
            ids2 = self.tokenizer(' ' + letter, add_special_tokens=False).input_ids
            if len(ids2) == 1:
                ids = ids2
            else:
                raise AssertionError(f'Letter {letter!r} tokenized to {len(ids)} or {len(ids2)} tokens; expected one token.')
        self._letter_token_cache[letter] = ids[0]
        return ids[0]

    @torch.no_grad()
    def score_letters_batched(self, prompts: List[Tuple[str, str]], letters_per_prompt: List[List[str]], priming_suffix: str = '') -> List[Dict[str, float]]:
        results = []
        full_prefixes = [self._build_prefix(sys_p, user_p) + priming_suffix for sys_p, user_p in prompts]
        for start in range(0, len(full_prefixes), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(full_prefixes))
            batch_texts = full_prefixes[start:end]
            batch_letters = letters_per_prompt[start:end]
            enc = self.tokenizer(
                batch_texts,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=MAX_MODEL_LEN,
            )
            first_device = next(self.model.parameters()).device
            enc = {k: v.to(first_device) for k, v in enc.items()}
            out = self.model(**enc)
            logits = out.logits
            # With left padding, the last prompt token is at index -1 for every sequence.
            for i in range(logits.shape[0]):
                next_logits = logits[i, -1, :]
                lp = torch.log_softmax(next_logits.float(), dim=-1)
                row_result = {}
                for letter in batch_letters[i]:
                    tok_id = self._single_token_id(letter)
                    row_result[letter] = float(lp[tok_id].detach().cpu())
                results.append(row_result)
            del enc, out, logits
            cleanup_memory()
        return results


In [ ]:
# =====================================================================
# Load Stage 1 scored rows and choose the shared ablation subset
# =====================================================================

def load_stage1_target(target_col: str, stage1_out_dir: Path = STAGE1_FRAME_DIR) -> tuple[pd.DataFrame, dict]:
    scored_path = stage1_out_dir / f'phase1_scored_{target_col}.parquet'
    meta_path = scored_path.with_suffix('.meta.json')
    if not scored_path.exists():
        raise FileNotFoundError(f'Missing Stage 1 scored parquet: {scored_path}')
    scored = pd.read_parquet(scored_path).copy()
    meta = {}
    if meta_path.exists():
        with open(meta_path) as f:
            meta = json.load(f)
    return scored, meta


def infer_classes(scored: pd.DataFrame, meta: dict, target_col: str) -> list[str]:
    if meta.get('class_labels'):
        return list(meta['class_labels'])
    prob_cols = [c for c in scored.columns if c.startswith('prob_') and not c.startswith('prob_raw_') and not c.startswith('prob_deb_')]
    classes = [c.replace('prob_', '') for c in prob_cols]
    if not classes:
        true_col = f'true_{target_col}'
        classes = sorted(scored[true_col].dropna().unique())
    return classes


def normalize_focus_groups(classes_by_target):
    """Make AGE_BINS focus labels match whatever dash style the data uses."""
    out = {}
    for target, groups in FOCUS_GROUPS.items():
        classes = classes_by_target.get(target, [])
        norm_to_actual = {normalize_label(c): c for c in classes}
        new_groups = []
        for g in groups:
            new_groups.append([norm_to_actual.get(normalize_label(x), x) for x in g])
        out[target] = new_groups
    return out


def make_letter_map(classes: list[str], seed: int) -> dict[str, str]:
    rng = np.random.default_rng(seed)
    letters = list(string.ascii_uppercase[:len(classes)])
    shuffled = list(classes)
    rng.shuffle(shuffled)
    return {L: c for L, c in zip(letters, shuffled)}


def canonicalize_scored_rows(scored: pd.DataFrame, target_col: str, classes: list[str]) -> pd.DataFrame:
    out = scored.copy()
    true_col = f'true_{target_col}'
    if true_col not in out.columns:
        if target_col in out.columns:
            out[true_col] = out[target_col]
        else:
            raise KeyError(f'Missing true label column for {target_col}: expected {true_col}')
    if 'row_uid' not in out.columns:
        if 'row_seed' in out.columns:
            out['row_uid'] = out['row_seed'].astype(str)
        else:
            out['row_uid'] = np.arange(len(out)).astype(str)
    if 'row_seed' not in out.columns:
        out['row_seed'] = np.arange(len(out)) + RANDOM_SEED
    if 'hajek_weight' not in out.columns:
        if 'PERWT' in out.columns:
            out['hajek_weight'] = pd.to_numeric(out['PERWT'], errors='coerce').fillna(1.0)
        elif 'eval_weight' in out.columns:
            out['hajek_weight'] = pd.to_numeric(out['eval_weight'], errors='coerce').fillna(1.0)
        else:
            out['hajek_weight'] = 1.0
    if 'letter_map_json' not in out.columns:
        print(f'No letter_map_json found for {target_col}; creating deterministic maps from row_seed.')
        out['letter_map_json'] = [json.dumps(make_letter_map(classes, int(s))) for s in out['row_seed']]
    out = out[out[true_col].isin(classes)].copy()
    out = out.dropna(subset=[true_col]).copy()
    return out


def choose_balanced_ablation_subset(scored: pd.DataFrame, target_col: str, classes: list[str], n_per_class: int, seed: int) -> pd.DataFrame:
    true_col = f'true_{target_col}'
    pool = scored.copy()
    if USE_INFERENCE_ROWS_ONLY and 'pride_role' in pool.columns:
        inf = pool[pool['pride_role'] == 'inference']
        if len(inf) > 0:
            pool = inf.copy()
    chunks = []
    for i, cls in enumerate(classes):
        cls_pool = pool[pool[true_col] == cls]
        if len(cls_pool) == 0:
            print(f'WARNING: no rows for {target_col}={cls!r}; skipping this class.')
            continue
        n = min(n_per_class, len(cls_pool))
        chunks.append(cls_pool.sample(n=n, random_state=seed + i, replace=False))
    if not chunks:
        raise ValueError(f'No rows available for target {target_col}.')
    subset = pd.concat(chunks, ignore_index=False).sample(frac=1, random_state=seed).reset_index(drop=True)
    subset['ablation_sample_row_id'] = [f'{target_col}_{i}' for i in range(len(subset))]
    return subset


def prepare_shared_ablation_samples():
    samples, meta_by_target, classes_by_target, context_by_target = {}, {}, {}, {}
    for target in ABLATION_TARGETS:
        scored, meta = load_stage1_target(target)
        classes = infer_classes(scored, meta, target)
        scored = canonicalize_scored_rows(scored, target, classes)
        subset = choose_balanced_ablation_subset(scored, target, classes, N_ABLATION_PER_CLASS, RANDOM_SEED)
        context_features = context_features_for(subset, target)
        samples[target] = subset
        meta_by_target[target] = meta
        classes_by_target[target] = classes
        context_by_target[target] = context_features
        sample_dir = TABLES_DIR / 'shared_samples'
        sample_dir.mkdir(parents=True, exist_ok=True)
        subset.to_parquet(sample_dir / f'shared_ablation_sample_{target}.parquet', index=False)
        print('\n' + '='*80)
        print(target)
        print('classes:', classes)
        print('sample rows:', len(subset))
        print('context features:', context_features)
        print('true class counts:', subset[f'true_{target}'].value_counts().to_dict())
    return samples, meta_by_target, classes_by_target, context_by_target

samples_by_target, meta_by_target, classes_by_target, context_by_target = prepare_shared_ablation_samples()
FOCUS_GROUPS = normalize_focus_groups(classes_by_target)
save_json({'classes_by_target': classes_by_target, 'context_by_target': context_by_target, 'focus_groups': FOCUS_GROUPS}, TABLES_DIR / 'shared_sample_metadata.json')
print('Normalized focus groups:', FOCUS_GROUPS)


In [ ]:
# =====================================================================
# Scoring baseline + leave-one-feature-out conditions
# =====================================================================

def get_letter_prior(meta: dict) -> Optional[dict[str, float]]:
    if USE_STAGE1_LETTER_PRIOR and meta.get('letter_log_prior'):
        return {str(k): float(v) for k, v in meta['letter_log_prior'].items()}
    return None


def score_condition(scorer, rows, target_col, classes, condition, drop_features, out_path, letter_log_prior=None):
    if out_path.exists():
        print('Loading cached:', out_path)
        return pd.read_parquet(out_path)

    true_col = f'true_{target_col}'
    pred_col = f'pred_{target_col}'
    letters = list(string.ascii_uppercase[:len(classes)])
    prompts, letters_per_prompt, letter_maps, user_prompts, answer_options_jsons = [], [], [], [], []

    for _, row in rows.iterrows():
        lm = json.loads(row['letter_map_json'])
        lettered_options = [(L, _display_label(lm[L])) for L in letters]
        user_prompt = build_user_prompt(row, target_col, lettered_options, drop_features=drop_features)
        prompts.append((SYSTEM_PROMPT, user_prompt))
        letters_per_prompt.append(letters)
        letter_maps.append(lm)
        user_prompts.append(user_prompt)
        answer_options_jsons.append(json.dumps([{'letter': L, 'class': lm[L]} for L in letters]))

    all_lp = []
    for start in tqdm(range(0, len(prompts), BATCH_SIZE), desc=f'{scorer.model_label} {target_col} {condition}'):
        end = min(start + BATCH_SIZE, len(prompts))
        all_lp.extend(scorer.score_letters_batched(prompts[start:end], letters_per_prompt[start:end], priming_suffix=PRIMING_SUFFIX))
        if (start // BATCH_SIZE + 1) % 25 == 0:
            print(f'  scored {end}/{len(prompts)}')

    records = []
    for i, (raw_letter_lp, lm) in enumerate(zip(all_lp, letter_maps)):
        row = rows.iloc[i]
        class_to_letter = {c: L for L, c in lm.items()}
        cls_letters = [class_to_letter[c] for c in classes]
        ll_class = np.array([raw_letter_lp.get(L, -100.0) for L in cls_letters], dtype=np.float64)
        p_raw = softmax(ll_class)
        rec = {
            'model_label': scorer.model_label,
            'model_name_or_path': scorer.model_name_or_path,
            'target': target_col,
            'condition': condition,
            'drop_features_json': json.dumps(drop_features),
            'ablation_sample_row_id': row.get('ablation_sample_row_id'),
            'row_uid': row.get('row_uid'),
            'row_seed': row.get('row_seed', np.nan),
            true_col: row[true_col],
            'hajek_weight': row.get('hajek_weight', 1.0),
            'classes_json': json.dumps(classes),
            'letter_map_json': json.dumps(lm),
            'letter_logprob_json': json.dumps(raw_letter_lp),
            'answer_options_json': answer_options_jsons[i],
            'system_prompt': SYSTEM_PROMPT,
            'user_prompt': user_prompts[i],
        }
        for col in FEATURE_LABELS:
            if col in row.index:
                rec[col] = row[col]
        for L in letters:
            rec[f'logprob_letter_{L}'] = raw_letter_lp.get(L, np.nan)
        for j, c in enumerate(classes):
            rec[f'logprob_raw_{c}'] = float(ll_class[j])
            rec[f'prob_raw_{c}'] = float(p_raw[j])
        if letter_log_prior is not None:
            prior_at_letter = np.array([letter_log_prior.get(L, 0.0) for L in cls_letters], dtype=np.float64)
            p = softmax(ll_class - prior_at_letter)
            rec['used_letter_prior'] = True
        else:
            p = p_raw
            rec['used_letter_prior'] = False
        for j, c in enumerate(classes):
            rec[f'prob_{c}'] = float(p[j])
            rec[f'logprob_{c}'] = float(np.log(max(p[j], 1e-300)))
        pred = classes[int(np.argmax([rec[f'prob_{c}'] for c in classes]))]
        rec[pred_col] = pred
        rec['p_true'] = float(rec[f'prob_{rec[true_col]}'])
        rec['correct'] = int(pred == rec[true_col])
        records.append(rec)

    out = pd.DataFrame(records)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out.to_parquet(out_path, index=False)
    print('Saved:', out_path)
    return out


def score_all_for_model(model_label: str, model_path: str):
    scorer = LetterScorer(model_path, model_label=model_label)
    outputs = {}
    try:
        for target in ABLATION_TARGETS:
            rows = samples_by_target[target]
            classes = classes_by_target[target]
            meta = meta_by_target[target]
            letter_prior = get_letter_prior(meta)
            context_features = context_by_target[target]
            target_outputs = {}
            model_target_dir = SCORES_DIR / safe_name(model_label) / target
            model_target_dir.mkdir(parents=True, exist_ok=True)
            conditions = [('baseline_full_profile', [])] + [(f'drop_{feat}', [feat]) for feat in context_features]
            for condition, drops in conditions:
                out_path = model_target_dir / f'{safe_name(condition)}.parquet'
                target_outputs[condition] = score_condition(scorer, rows, target, classes, condition, drops, out_path, letter_prior)
            outputs[target] = target_outputs
    finally:
        scorer.close()
    return outputs


In [ ]:
# =====================================================================
# Run scoring for base and fine-tuned models
# =====================================================================
model_runs = [(BASE_RUN_NAME, BASE_MODEL_PATH), (FT_RUN_NAME, FT_MODEL_PATH)]
all_model_outputs = {}
for model_label, model_path in model_runs:
    print('\n' + '='*100)
    print('SCORING MODEL:', model_label, model_path)
    print('='*100)
    all_model_outputs[model_label] = score_all_for_model(model_label, model_path)

all_scored_parts = []
for model_label in all_model_outputs:
    for target in all_model_outputs[model_label]:
        for condition, df in all_model_outputs[model_label][target].items():
            all_scored_parts.append(df)
all_ablation_scores = pd.concat(all_scored_parts, ignore_index=True)
all_ablation_scores.to_parquet(SCORES_DIR / 'all_feature_ablation_scores.parquet', index=False)
print('Saved:', SCORES_DIR / 'all_feature_ablation_scores.parquet')
display(all_ablation_scores.head())


In [ ]:
# =====================================================================
# Metrics: baseline fidelity, class shifts, feature-driver summaries
# =====================================================================

def focus_label(focus_classes):
    return '+'.join(focus_classes)


def brier_for_rows(df, target_col, classes):
    true_col = f'true_{target_col}'
    P = df[[f'prob_{c}' for c in classes]].to_numpy(dtype=float)
    y = np.array([classes.index(v) for v in df[true_col].values])
    Y = np.eye(len(classes))[y]
    return np.sum((P - Y) ** 2, axis=1)


def condition_metrics(df, target_col, classes):
    true_col = f'true_{target_col}'
    pred_col = f'pred_{target_col}'
    w = df['hajek_weight'].to_numpy(dtype=float) if 'hajek_weight' in df.columns else np.ones(len(df))
    correct = (df[pred_col].values == df[true_col].values).astype(float)
    p_true = df['p_true'].astype(float).clip(1e-12, 1.0).values
    brier = brier_for_rows(df, target_col, classes)
    return {
        'n': len(df),
        'accuracy': weighted_mean(correct, w),
        'mean_p_true': weighted_mean(p_true, w),
        'brier': weighted_mean(brier, w),
        'log_loss': weighted_mean(-np.log(p_true), w),
    }


def focus_metrics(df, target_col, classes, focus_classes):
    true_col = f'true_{target_col}'
    pred_col = f'pred_{target_col}'
    w = df['hajek_weight'].to_numpy(dtype=float) if 'hajek_weight' in df.columns else np.ones(len(df))
    p_focus = df[[f'prob_{c}' for c in focus_classes]].sum(axis=1).to_numpy(dtype=float)
    true_focus = df[true_col].isin(focus_classes).to_numpy()
    pred_focus = df[pred_col].isin(focus_classes).to_numpy()
    nonfocus = ~true_focus
    return {
        'focus_group': focus_label(focus_classes),
        'true_focus_share': weighted_mean(true_focus.astype(float), w),
        'mean_prob_focus': weighted_mean(p_focus, w),
        'pred_focus_rate': weighted_mean(pred_focus.astype(float), w),
        'overprob_focus': weighted_mean(p_focus, w) - weighted_mean(true_focus.astype(float), w),
        'overpred_focus': weighted_mean(pred_focus.astype(float), w) - weighted_mean(true_focus.astype(float), w),
        'share_p_focus_gt_0.9': weighted_mean((p_focus > 0.9).astype(float), w),
        'nonfocus_mean_prob_focus': weighted_mean(p_focus[nonfocus], w[nonfocus]) if nonfocus.sum() else np.nan,
        'false_positive_rate_focus': weighted_mean(pred_focus[nonfocus].astype(float), w[nonfocus]) if nonfocus.sum() else np.nan,
    }


def class_shift_table(base, cond, target_col, classes):
    true_col = f'true_{target_col}'
    pred_col = f'pred_{target_col}'
    key = 'ablation_sample_row_id'
    b = base[[key, true_col, pred_col, 'hajek_weight'] + [f'prob_{c}' for c in classes]].copy()
    c = cond[[key, pred_col] + [f'prob_{c}' for c in classes]].copy()
    c = c.rename(columns={pred_col: f'{pred_col}_condition', **{f'prob_{cl}': f'prob_{cl}_condition' for cl in classes}})
    df = b.merge(c, on=key, how='inner')
    w = df['hajek_weight'].to_numpy(dtype=float)
    rows = []
    for cl in classes:
        true_mask = df[true_col].eq(cl).to_numpy()
        rows.append({
            'target': target_col,
            'class': cl,
            'n': len(df),
            'population_share': weighted_mean(true_mask.astype(float), w),
            'baseline_mean_prob': weighted_mean(df[f'prob_{cl}'], w),
            'condition_mean_prob': weighted_mean(df[f'prob_{cl}_condition'], w),
            'delta_mean_prob_condition_minus_baseline': weighted_mean(df[f'prob_{cl}_condition'] - df[f'prob_{cl}'], w),
            'baseline_pred_share': weighted_mean(df[pred_col].eq(cl).astype(float), w),
            'condition_pred_share': weighted_mean(df[f'{pred_col}_condition'].eq(cl).astype(float), w),
            'delta_pred_share_condition_minus_baseline': weighted_mean(df[f'{pred_col}_condition'].eq(cl).astype(float) - df[pred_col].eq(cl).astype(float), w),
        })
    return pd.DataFrame(rows)


def sensitivity_metrics(base, cond, target_col, classes):
    key = 'ablation_sample_row_id'
    b = base[[key, f'true_{target_col}', f'pred_{target_col}', 'hajek_weight'] + [f'prob_{c}' for c in classes]].copy()
    c = cond[[key, f'pred_{target_col}'] + [f'prob_{c}' for c in classes]].copy()
    c = c.rename(columns={f'pred_{target_col}': f'pred_{target_col}_condition', **{f'prob_{cl}': f'prob_{cl}_condition' for cl in classes}})
    df = b.merge(c, on=key, how='inner')
    w = df['hajek_weight'].to_numpy(dtype=float)
    P0 = df[[f'prob_{cl}' for cl in classes]].to_numpy(dtype=float)
    P1 = df[[f'prob_{cl}_condition' for cl in classes]].to_numpy(dtype=float)
    l1 = np.sum(np.abs(P1 - P0), axis=1)
    max_abs = np.max(np.abs(P1 - P0), axis=1)
    pred_changed = (df[f'pred_{target_col}'] != df[f'pred_{target_col}_condition']).astype(float).to_numpy()
    return {
        'mean_l1_prob_shift': weighted_mean(l1, w),
        'mean_max_abs_prob_shift': weighted_mean(max_abs, w),
        'pred_changed_rate': weighted_mean(pred_changed, w),
    }


def summarize_all_scores(all_scores):
    baseline_rows, condition_rows, driver_rows, shift_parts, sensitivity_rows = [], [], [], [], []
    for model_label in all_scores['model_label'].unique():
        for target in ABLATION_TARGETS:
            classes = classes_by_target[target]
            target_scores = all_scores[(all_scores['model_label'] == model_label) & (all_scores['target'] == target)].copy()
            baseline = target_scores[target_scores['condition'] == 'baseline_full_profile'].copy()
            if baseline.empty:
                continue
            base_metrics = condition_metrics(baseline, target, classes)
            baseline_rows.append({'model_label': model_label, 'target': target, 'condition': 'baseline_full_profile', **base_metrics})
            condition_rows.append({'model_label': model_label, 'target': target, 'condition': 'baseline_full_profile', 'ablation_feature': None, **base_metrics})
            focus_groups = FOCUS_GROUPS.get(target, [[c] for c in classes])
            for condition in sorted(target_scores['condition'].unique()):
                if condition == 'baseline_full_profile':
                    continue
                cond = target_scores[target_scores['condition'] == condition].copy()
                ablation_feature = condition.replace('drop_', '', 1)
                cond_metrics = condition_metrics(cond, target, classes)
                condition_rows.append({'model_label': model_label, 'target': target, 'condition': condition, 'ablation_feature': ablation_feature, **cond_metrics})
                sensitivity_rows.append({'model_label': model_label, 'target': target, 'condition': condition, 'ablation_feature': ablation_feature, **sensitivity_metrics(baseline, cond, target, classes)})
                shift = class_shift_table(baseline, cond, target, classes)
                shift['model_label'] = model_label
                shift['condition'] = condition
                shift['ablation_feature'] = ablation_feature
                shift_parts.append(shift)
                for fg in focus_groups:
                    bm = focus_metrics(baseline, target, classes, fg)
                    cm = focus_metrics(cond, target, classes, fg)
                    row = {
                        'model_label': model_label,
                        'target': target,
                        'condition': condition,
                        'ablation_feature': ablation_feature,
                        'focus_group': bm['focus_group'],
                        'true_focus_share': bm['true_focus_share'],
                        'baseline_mean_prob_focus': bm['mean_prob_focus'],
                        'condition_mean_prob_focus': cm['mean_prob_focus'],
                        'reduction_mean_prob_focus': bm['mean_prob_focus'] - cm['mean_prob_focus'],
                        'baseline_pred_focus_rate': bm['pred_focus_rate'],
                        'condition_pred_focus_rate': cm['pred_focus_rate'],
                        'reduction_pred_focus_rate': bm['pred_focus_rate'] - cm['pred_focus_rate'],
                        'baseline_overprob_focus': bm['overprob_focus'],
                        'condition_overprob_focus': cm['overprob_focus'],
                        'reduction_overprob_focus': bm['overprob_focus'] - cm['overprob_focus'],
                        'baseline_share_p_focus_gt_0.9': bm['share_p_focus_gt_0.9'],
                        'condition_share_p_focus_gt_0.9': cm['share_p_focus_gt_0.9'],
                        'reduction_share_p_focus_gt_0.9': bm['share_p_focus_gt_0.9'] - cm['share_p_focus_gt_0.9'],
                        'baseline_nonfocus_mean_prob_focus': bm['nonfocus_mean_prob_focus'],
                        'condition_nonfocus_mean_prob_focus': cm['nonfocus_mean_prob_focus'],
                        'reduction_nonfocus_mean_prob_focus': bm['nonfocus_mean_prob_focus'] - cm['nonfocus_mean_prob_focus'],
                        'baseline_false_positive_rate_focus': bm['false_positive_rate_focus'],
                        'condition_false_positive_rate_focus': cm['false_positive_rate_focus'],
                        'reduction_false_positive_rate_focus': bm['false_positive_rate_focus'] - cm['false_positive_rate_focus'],
                    }
                    row['driver_score'] = row['reduction_overprob_focus'] + row['reduction_false_positive_rate_focus'] + row['reduction_nonfocus_mean_prob_focus']
                    driver_rows.append(row)
    baseline_metrics = pd.DataFrame(baseline_rows)
    condition_metrics_df = pd.DataFrame(condition_rows)
    driver_summary = pd.DataFrame(driver_rows).sort_values(['target', 'focus_group', 'model_label', 'driver_score'], ascending=[True, True, True, False])
    class_shift_summary = pd.concat(shift_parts, ignore_index=True) if shift_parts else pd.DataFrame()
    sensitivity_summary = pd.DataFrame(sensitivity_rows)
    return baseline_metrics, condition_metrics_df, driver_summary, class_shift_summary, sensitivity_summary

all_scores = pd.read_parquet(SCORES_DIR / 'all_feature_ablation_scores.parquet')
baseline_metrics, condition_metrics_df, driver_summary, class_shift_summary, sensitivity_summary = summarize_all_scores(all_scores)
for name, df in [
    ('baseline_model_metrics.csv', baseline_metrics),
    ('condition_metrics.csv', condition_metrics_df),
    ('driver_summary.csv', driver_summary),
    ('class_shift_summary.csv', class_shift_summary),
    ('ablation_sensitivity_summary.csv', sensitivity_summary),
]:
    df.to_csv(TABLES_DIR / name, index=False)

base_driver = driver_summary[driver_summary['model_label'] == BASE_RUN_NAME].copy()
ft_driver = driver_summary[driver_summary['model_label'] == FT_RUN_NAME].copy()
driver_compare = base_driver.merge(ft_driver, on=['target', 'focus_group', 'ablation_feature'], suffixes=('_base', '_ft'), how='inner')
for col in ['driver_score', 'reduction_overprob_focus', 'reduction_false_positive_rate_focus', 'reduction_nonfocus_mean_prob_focus']:
    driver_compare[f'{col}_delta_ft_minus_base'] = driver_compare[f'{col}_ft'] - driver_compare[f'{col}_base']
driver_compare.to_csv(TABLES_DIR / 'driver_compare_ft_minus_base.csv', index=False)

base_metrics = baseline_metrics[baseline_metrics['model_label'] == BASE_RUN_NAME].copy()
ft_metrics = baseline_metrics[baseline_metrics['model_label'] == FT_RUN_NAME].copy()
baseline_compare = base_metrics.merge(ft_metrics, on=['target', 'condition'], suffixes=('_base', '_ft'), how='inner')
for col in ['accuracy', 'mean_p_true']:
    baseline_compare[f'{col}_delta_ft_minus_base'] = baseline_compare[f'{col}_ft'] - baseline_compare[f'{col}_base']
for col in ['brier', 'log_loss']:
    baseline_compare[f'{col}_delta_ft_minus_base'] = baseline_compare[f'{col}_ft'] - baseline_compare[f'{col}_base']
    baseline_compare[f'{col}_improvement_ft_vs_base'] = baseline_compare[f'{col}_base'] - baseline_compare[f'{col}_ft']
baseline_compare.to_csv(TABLES_DIR / 'baseline_compare_ft_minus_base.csv', index=False)
print('Saved metric tables to:', TABLES_DIR)
display(baseline_compare)
display(driver_compare.sort_values('driver_score_delta_ft_minus_base', ascending=False).head(20))


In [ ]:
# =====================================================================
# Plots: baseline metrics, driver bars, FT-minus-base changes, sensitivity
# =====================================================================

def savefig(path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, dpi=180, bbox_inches='tight')
    plt.show()
    print('Saved:', path)

# 1) Baseline full-profile metrics: base vs FT.
for metric, ylabel in [
    ('accuracy', 'Accuracy'),
    ('brier', 'Brier score (lower is better)'),
    ('log_loss', 'Log loss (lower is better)'),
    ('mean_p_true', 'Mean probability assigned to true class'),
]:
    df = baseline_metrics.copy()
    targets = ABLATION_TARGETS
    models = [BASE_RUN_NAME, FT_RUN_NAME]
    x = np.arange(len(targets))
    width = 0.35
    plt.figure(figsize=(9, 5))
    for i, model in enumerate(models):
        vals = []
        for target in targets:
            sub = df[(df['model_label'] == model) & (df['target'] == target)]
            vals.append(float(sub[metric].iloc[0]) if len(sub) else np.nan)
        plt.bar(x + (i - 0.5) * width, vals, width, label=model)
    plt.xticks(x, targets, rotation=20, ha='right')
    plt.ylabel(ylabel)
    plt.title(f'Baseline full-profile {metric}: base vs fine-tuned')
    plt.legend()
    plt.tight_layout()
    savefig(PLOTS_DIR / f'baseline_metric_{metric}.png')

# 2) Driver bars by model for each target/focus.
for (target, focus), dfp in driver_summary.groupby(['target', 'focus_group']):
    rank = dfp.groupby('ablation_feature')['driver_score'].max().sort_values(ascending=True)
    features = list(rank.tail(12).index)
    plot_df = dfp[dfp['ablation_feature'].isin(features)].copy()
    x = np.arange(len(features))
    width = 0.35
    plt.figure(figsize=(11, max(5, 0.45 * len(features))))
    for i, model in enumerate([BASE_RUN_NAME, FT_RUN_NAME]):
        vals = []
        for feat in features:
            sub = plot_df[(plot_df['model_label'] == model) & (plot_df['ablation_feature'] == feat)]
            vals.append(float(sub['driver_score'].iloc[0]) if len(sub) else np.nan)
        plt.barh(x + (i - 0.5) * width, vals, height=width, label=model)
    plt.yticks(x, features)
    plt.axvline(0, linewidth=1)
    plt.xlabel('Driver score: positive = dropping feature reduced focal-class attraction')
    plt.title(f'Feature drivers for {target} -> {focus}: base vs fine-tuned')
    plt.legend()
    plt.tight_layout()
    savefig(PLOTS_DIR / f'driver_bar_compare_{safe_name(target)}_{safe_name(focus)}.png')

# 3) FT-minus-base driver deltas.
for (target, focus), dfp in driver_compare.groupby(['target', 'focus_group']):
    plot_df = dfp.sort_values('driver_score_delta_ft_minus_base', ascending=True)
    plt.figure(figsize=(10, max(4, 0.35 * len(plot_df))))
    plt.barh(plot_df['ablation_feature'], plot_df['driver_score_delta_ft_minus_base'])
    plt.axvline(0, linewidth=1)
    plt.xlabel('FT driver score - Base driver score')
    plt.title(f'Change in feature reliance after fine-tuning: {target} -> {focus}')
    plt.tight_layout()
    savefig(PLOTS_DIR / f'driver_delta_ft_minus_base_{safe_name(target)}_{safe_name(focus)}.png')

# 4) Ablation sensitivity: how much predictions move when any one feature is dropped.
if not sensitivity_summary.empty:
    sens_agg = sensitivity_summary.groupby(['model_label', 'target'])[
        ['mean_l1_prob_shift', 'mean_max_abs_prob_shift', 'pred_changed_rate']
    ].mean().reset_index()
    sens_agg.to_csv(TABLES_DIR / 'ablation_sensitivity_by_model_target.csv', index=False)
    for metric in ['mean_l1_prob_shift', 'mean_max_abs_prob_shift', 'pred_changed_rate']:
        x = np.arange(len(ABLATION_TARGETS))
        width = 0.35
        plt.figure(figsize=(9, 5))
        for i, model in enumerate([BASE_RUN_NAME, FT_RUN_NAME]):
            vals = []
            for target in ABLATION_TARGETS:
                sub = sens_agg[(sens_agg['model_label'] == model) & (sens_agg['target'] == target)]
                vals.append(float(sub[metric].iloc[0]) if len(sub) else np.nan)
            plt.bar(x + (i - 0.5) * width, vals, width, label=model)
        plt.xticks(x, ABLATION_TARGETS, rotation=20, ha='right')
        plt.ylabel(metric)
        plt.title(f'Average ablation sensitivity: {metric}')
        plt.legend()
        plt.tight_layout()
        savefig(PLOTS_DIR / f'ablation_sensitivity_{metric}.png')

# 5) Heatmap-like plot of driver deltas.
for target in ABLATION_TARGETS:
    sub = driver_compare[driver_compare['target'] == target].copy()
    if sub.empty:
        continue
    focus = sub['focus_group'].iloc[0]
    pivot = sub.pivot_table(index='ablation_feature', values='driver_score_delta_ft_minus_base', aggfunc='mean')
    pivot = pivot.sort_values('driver_score_delta_ft_minus_base')
    plt.figure(figsize=(8, max(4, 0.35 * len(pivot))))
    plt.imshow(pivot[['driver_score_delta_ft_minus_base']].values, aspect='auto')
    plt.yticks(np.arange(len(pivot.index)), pivot.index)
    plt.xticks([0], ['FT - Base'])
    plt.colorbar(label='Driver-score delta')
    plt.title(f'Driver-score delta heatmap: {target} -> {focus}')
    plt.tight_layout()
    savefig(PLOTS_DIR / f'heatmap_driver_delta_{safe_name(target)}.png')


In [ ]:
# =====================================================================
# Response distribution plots for baseline and selected ablations
# =====================================================================

def unweighted_class_distribution(df, label_col, classes):
    denom = len(df)
    return pd.DataFrame({
        'class': classes,
        'share': [float(df[label_col].eq(c).sum() / denom) if denom else np.nan for c in classes],
        'count': [int(df[label_col].eq(c).sum()) for c in classes],
        'n': denom,
    })


def mean_probability_distribution(df, classes):
    return pd.DataFrame({
        'class': classes,
        'share': [float(pd.to_numeric(df[f'prob_{c}'], errors='coerce').mean()) for c in classes],
        'count': np.nan,
        'n': len(df),
    })


def grouped_bar(plot_df, title, ylabel, path):
    classes = list(plot_df['class'].unique())
    series = list(plot_df['series'].unique())
    x = np.arange(len(classes))
    width = 0.8 / max(len(series), 1)
    plt.figure(figsize=(max(8, len(classes)*1.3), 5))
    for i, s in enumerate(series):
        vals = []
        sub = plot_df[plot_df['series'] == s]
        for c in classes:
            v = sub.loc[sub['class'] == c, 'share']
            vals.append(float(v.iloc[0]) if len(v) else np.nan)
        plt.bar(x + (i - (len(series)-1)/2)*width, vals, width, label=s)
    plt.xticks(x, classes, rotation=25, ha='right')
    plt.ylim(0, max(1.0, np.nanmax(plot_df['share'].values)*1.15))
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    savefig(path)

all_scores = pd.read_parquet(SCORES_DIR / 'all_feature_ablation_scores.parquet')
bar_records = []
for target in ABLATION_TARGETS:
    classes = classes_by_target[target]
    true_col = f'true_{target}'
    pred_col = f'pred_{target}'
    target_compare = driver_compare[driver_compare['target'] == target].copy()
    top_feats = list(target_compare.assign(abs_delta=lambda d: d['driver_score_delta_ft_minus_base'].abs())
                     .sort_values('abs_delta', ascending=False)['ablation_feature'].head(3))
    conditions = ['baseline_full_profile'] + [f'drop_{f}' for f in top_feats]
    for condition in conditions:
        sub = all_scores[(all_scores['target'] == target) & (all_scores['condition'] == condition)].copy()
        if sub.empty:
            continue
        true_sub = sub.drop_duplicates(['ablation_sample_row_id', 'target', 'condition'])
        true_dist = unweighted_class_distribution(true_sub, true_col, classes)
        true_dist['series'] = 'True sampled rows'
        pred_parts = [true_dist]
        prob_parts = []
        for model in [BASE_RUN_NAME, FT_RUN_NAME]:
            m = sub[sub['model_label'] == model].copy()
            if m.empty:
                continue
            pred_dist = unweighted_class_distribution(m, pred_col, classes)
            pred_dist['series'] = f'{model} top pred'
            pred_parts.append(pred_dist)
            prob_dist = mean_probability_distribution(m, classes)
            prob_dist['series'] = f'{model} mean prob'
            prob_parts.append(prob_dist)
        pred_plot = pd.concat(pred_parts, ignore_index=True)
        pred_plot['target'] = target
        pred_plot['condition'] = condition
        pred_plot['plot_type'] = 'top_prediction_distribution'
        bar_records.append(pred_plot)
        grouped_bar(
            pred_plot,
            title=f'True vs top-predicted distribution: {target} / {condition}',
            ylabel='Share of sampled rows',
            path=PLOTS_DIR / f'dist_top_pred_{safe_name(target)}_{safe_name(condition)}.png',
        )
        if prob_parts:
            prob_plot = pd.concat(prob_parts, ignore_index=True)
            prob_plot['target'] = target
            prob_plot['condition'] = condition
            prob_plot['plot_type'] = 'mean_probability_distribution'
            bar_records.append(prob_plot)
            grouped_bar(
                prob_plot,
                title=f'Mean predicted probability distribution: {target} / {condition}',
                ylabel='Mean predicted probability',
                path=PLOTS_DIR / f'dist_mean_prob_{safe_name(target)}_{safe_name(condition)}.png',
            )
bar_summary = pd.concat(bar_records, ignore_index=True) if bar_records else pd.DataFrame()
bar_summary.to_csv(TABLES_DIR / 'distribution_bar_summary.csv', index=False)
display(bar_summary.head(30))


In [ ]:
# =====================================================================
# Final manifest of outputs
# =====================================================================
manifest = []
for p in sorted(OUT_DIR.rglob('*')):
    if p.is_file():
        manifest.append({'path': str(p), 'size_mb': p.stat().st_size / 1e6})
manifest_df = pd.DataFrame(manifest)
manifest_df.to_csv(OUT_DIR / 'output_manifest.csv', index=False)
display(manifest_df.tail(100))
print('Done. Output directory:', OUT_DIR)


# Ablation group experiments

In [ ]:
# =====================================================================
# ADD-ON: grouped feature ablation experiment
# =====================================================================
# Paste this at the END of the notebook after the individual-drop run finishes.
#
# This does NOT overwrite your existing individual-feature ablation outputs.
# It creates separate folders:
#   OUT_DIR/scores_grouped/
#   OUT_DIR/tables_grouped/
#   OUT_DIR/plots_grouped/
#
# It reuses the existing baseline_full_profile scores from SCORES_DIR when available,
# then scores only grouped ablation conditions.

GROUP_SCORES_DIR = OUT_DIR / "scores_grouped"
GROUP_TABLES_DIR = OUT_DIR / "tables_grouped"
GROUP_PLOTS_DIR = OUT_DIR / "plots_grouped"

for p in [GROUP_SCORES_DIR, GROUP_TABLES_DIR, GROUP_PLOTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Grouped ablation output dirs:")
print("  GROUP_SCORES_DIR:", GROUP_SCORES_DIR)
print("  GROUP_TABLES_DIR:", GROUP_TABLES_DIR)
print("  GROUP_PLOTS_DIR:", GROUP_PLOTS_DIR)


# ---------------------------------------------------------------------
# Define grouped drops
# ---------------------------------------------------------------------
def make_grouped_drop_conditions(target_col: str, context_features: list[str]) -> list[tuple[str, list[str]]]:
    """
    Grouped feature ablations.
    The target column itself is never included in a drop list.
    Only features actually present in the context_features list are included.
    """
    context_features = list(context_features)

    raw_groups = {
        "drop_core_demographics": [
            "SEX",
            "AGE_BINS",
            "RACE_ETHNICITY",
        ],

        "drop_socioeconomic": [
            "EDUCATION_LEVEL",
            "INCTOT_BINS",
            "HEALTH_INSURANCE",
            "OWNERSHP_STATUS",
        ],

        "drop_geography_language": [
            "STATE_NAME",
            "ENGLISH_STATUS",
        ],

        "drop_family_household": [
            "MARITAL_STATUS",
            "NCHILD",
        ],

        "drop_work_commute": [
            "EMPLOYMENT_STATUS",
            "WORKER_CLASS",
            "UHRSWORK",
            "TRANSPORT_METHOD",
        ],

        "drop_all_context": [
            f for f in context_features
        ],
    }

    conditions = []
    for group_name, feats in raw_groups.items():
        drops = [
            f for f in feats
            if f in context_features and f != target_col
        ]

        # Avoid empty groups.
        if len(drops) > 0:
            conditions.append((group_name, drops))

    return conditions


# Preview grouped drops for each target.
for target in ABLATION_TARGETS:
    print("\n" + "=" * 80)
    print("Target:", target)
    for condition, drops in make_grouped_drop_conditions(target, context_by_target[target]):
        print(f"  {condition}: {drops}")

In [ ]:
# =====================================================================
# Run grouped ablation scoring
# =====================================================================

def load_or_score_grouped_for_model(model_label: str, model_path: str):
    scorer = LetterScorer(model_path, model_label=model_label)
    outputs = {}

    try:
        for target in ABLATION_TARGETS:
            rows = samples_by_target[target]
            classes = classes_by_target[target]
            meta = meta_by_target[target]
            letter_prior = get_letter_prior(meta)
            context_features = context_by_target[target]

            target_outputs = {}
            model_target_dir = GROUP_SCORES_DIR / safe_name(model_label) / target
            model_target_dir.mkdir(parents=True, exist_ok=True)

            # Reuse already-scored baseline from the individual-feature run if available.
            old_baseline_path = SCORES_DIR / safe_name(model_label) / target / "baseline_full_profile.parquet"
            grouped_baseline_path = model_target_dir / "baseline_full_profile.parquet"

            if grouped_baseline_path.exists():
                print("Loading grouped baseline:", grouped_baseline_path)
                target_outputs["baseline_full_profile"] = pd.read_parquet(grouped_baseline_path)
            elif old_baseline_path.exists():
                print("Reusing existing baseline:", old_baseline_path)
                baseline_df = pd.read_parquet(old_baseline_path)
                baseline_df.to_parquet(grouped_baseline_path, index=False)
                target_outputs["baseline_full_profile"] = baseline_df
            else:
                print("Baseline not found; scoring baseline:", model_label, target)
                target_outputs["baseline_full_profile"] = score_condition(
                    scorer,
                    rows,
                    target,
                    classes,
                    "baseline_full_profile",
                    [],
                    grouped_baseline_path,
                    letter_prior,
                )

            group_conditions = make_grouped_drop_conditions(target, context_features)

            print("\n" + "=" * 80)
            print(f"Grouped ablations for model={model_label}, target={target}")
            for condition, drops in group_conditions:
                print(f"  {condition}: {drops}")

            for condition, drops in group_conditions:
                out_path = model_target_dir / f"{safe_name(condition)}.parquet"
                target_outputs[condition] = score_condition(
                    scorer,
                    rows,
                    target,
                    classes,
                    condition,
                    drops,
                    out_path,
                    letter_prior,
                )

            outputs[target] = target_outputs

    finally:
        scorer.close()

    return outputs


group_model_runs = [
    (BASE_RUN_NAME, BASE_MODEL_PATH),
    (FT_RUN_NAME, FT_MODEL_PATH),
]

group_model_outputs = {}

for model_label, model_path in group_model_runs:
    print("\n" + "=" * 100)
    print("GROUPED ABLATION SCORING MODEL:", model_label, model_path)
    print("=" * 100)

    group_model_outputs[model_label] = load_or_score_grouped_for_model(model_label, model_path)


# Collect grouped scores into one parquet.
group_scored_parts = []
for model_label in group_model_outputs:
    for target in group_model_outputs[model_label]:
        for condition, df in group_model_outputs[model_label][target].items():
            group_scored_parts.append(df)

group_ablation_scores = pd.concat(group_scored_parts, ignore_index=True)
group_all_scores_path = GROUP_SCORES_DIR / "all_grouped_feature_ablation_scores.parquet"
group_ablation_scores.to_parquet(group_all_scores_path, index=False)

print("Saved grouped scores:", group_all_scores_path)
display(group_ablation_scores.head())

In [ ]:
# =====================================================================
# Grouped ablation metrics + plots
# =====================================================================

group_scores = pd.read_parquet(GROUP_SCORES_DIR / "all_grouped_feature_ablation_scores.parquet")

(
    grouped_baseline_metrics,
    grouped_condition_metrics_df,
    grouped_driver_summary,
    grouped_class_shift_summary,
    grouped_sensitivity_summary,
) = summarize_all_scores(group_scores)

# Save grouped metric tables.
for name, df in [
    ("grouped_baseline_model_metrics.csv", grouped_baseline_metrics),
    ("grouped_condition_metrics.csv", grouped_condition_metrics_df),
    ("grouped_driver_summary.csv", grouped_driver_summary),
    ("grouped_class_shift_summary.csv", grouped_class_shift_summary),
    ("grouped_ablation_sensitivity_summary.csv", grouped_sensitivity_summary),
]:
    path = GROUP_TABLES_DIR / name
    df.to_csv(path, index=False)
    print("Saved:", path)

# Compare grouped driver scores: FT minus base.
group_base_driver = grouped_driver_summary[grouped_driver_summary["model_label"] == BASE_RUN_NAME].copy()
group_ft_driver = grouped_driver_summary[grouped_driver_summary["model_label"] == FT_RUN_NAME].copy()

group_driver_compare = group_base_driver.merge(
    group_ft_driver,
    on=["target", "focus_group", "ablation_feature"],
    suffixes=("_base", "_ft"),
    how="inner",
)

for col in [
    "driver_score",
    "reduction_overprob_focus",
    "reduction_false_positive_rate_focus",
    "reduction_nonfocus_mean_prob_focus",
]:
    group_driver_compare[f"{col}_delta_ft_minus_base"] = (
        group_driver_compare[f"{col}_ft"] - group_driver_compare[f"{col}_base"]
    )

group_driver_compare_path = GROUP_TABLES_DIR / "grouped_driver_compare_ft_minus_base.csv"
group_driver_compare.to_csv(group_driver_compare_path, index=False)
print("Saved:", group_driver_compare_path)

# Compare baseline base vs FT for grouped run; mostly sanity check because baseline is shared/reused.
group_base_metrics = grouped_baseline_metrics[grouped_baseline_metrics["model_label"] == BASE_RUN_NAME].copy()
group_ft_metrics = grouped_baseline_metrics[grouped_baseline_metrics["model_label"] == FT_RUN_NAME].copy()

group_baseline_compare = group_base_metrics.merge(
    group_ft_metrics,
    on=["target", "condition"],
    suffixes=("_base", "_ft"),
    how="inner",
)

for col in ["accuracy", "mean_p_true"]:
    group_baseline_compare[f"{col}_delta_ft_minus_base"] = (
        group_baseline_compare[f"{col}_ft"] - group_baseline_compare[f"{col}_base"]
    )

for col in ["brier", "log_loss"]:
    group_baseline_compare[f"{col}_delta_ft_minus_base"] = (
        group_baseline_compare[f"{col}_ft"] - group_baseline_compare[f"{col}_base"]
    )
    group_baseline_compare[f"{col}_improvement_ft_vs_base"] = (
        group_baseline_compare[f"{col}_base"] - group_baseline_compare[f"{col}_ft"]
    )

group_baseline_compare_path = GROUP_TABLES_DIR / "grouped_baseline_compare_ft_minus_base.csv"
group_baseline_compare.to_csv(group_baseline_compare_path, index=False)
print("Saved:", group_baseline_compare_path)

display(group_baseline_compare)
display(group_driver_compare.sort_values("driver_score_delta_ft_minus_base", ascending=False))


# ---------------------------------------------------------------------
# Plot helpers for grouped outputs
# ---------------------------------------------------------------------
def savefig_grouped(path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved:", path)


# 1) Grouped driver bars by model.
for (target, focus), dfp in grouped_driver_summary.groupby(["target", "focus_group"]):
    plot_df = dfp.copy()
    groups = list(
        plot_df.groupby("ablation_feature")["driver_score"]
        .max()
        .sort_values(ascending=True)
        .index
    )

    x = np.arange(len(groups))
    width = 0.35

    plt.figure(figsize=(11, max(5, 0.55 * len(groups))))

    for i, model in enumerate([BASE_RUN_NAME, FT_RUN_NAME]):
        vals = []
        for group in groups:
            sub = plot_df[
                (plot_df["model_label"] == model)
                & (plot_df["ablation_feature"] == group)
            ]
            vals.append(float(sub["driver_score"].iloc[0]) if len(sub) else np.nan)

        plt.barh(
            x + (i - 0.5) * width,
            vals,
            height=width,
            label=model,
        )

    plt.yticks(x, groups)
    plt.axvline(0, linewidth=1)
    plt.xlabel("Driver score: positive = dropping group reduced focal-class attraction")
    plt.title(f"Grouped feature drivers for {target} -> {focus}: base vs fine-tuned")
    plt.legend()
    plt.tight_layout()

    savefig_grouped(
        GROUP_PLOTS_DIR / f"grouped_driver_bar_compare_{safe_name(target)}_{safe_name(focus)}.png"
    )


# 2) Grouped FT-minus-base driver deltas.
for (target, focus), dfp in group_driver_compare.groupby(["target", "focus_group"]):
    plot_df = dfp.sort_values("driver_score_delta_ft_minus_base", ascending=True)

    plt.figure(figsize=(10, max(4, 0.55 * len(plot_df))))
    plt.barh(plot_df["ablation_feature"], plot_df["driver_score_delta_ft_minus_base"])
    plt.axvline(0, linewidth=1)
    plt.xlabel("FT grouped driver score - Base grouped driver score")
    plt.title(f"Change in grouped feature reliance after fine-tuning: {target} -> {focus}")
    plt.tight_layout()

    savefig_grouped(
        GROUP_PLOTS_DIR / f"grouped_driver_delta_ft_minus_base_{safe_name(target)}_{safe_name(focus)}.png"
    )


# 3) Grouped ablation sensitivity.
if not grouped_sensitivity_summary.empty:
    group_sens_agg = (
        grouped_sensitivity_summary
        .groupby(["model_label", "target"])[
            ["mean_l1_prob_shift", "mean_max_abs_prob_shift", "pred_changed_rate"]
        ]
        .mean()
        .reset_index()
    )

    group_sens_agg_path = GROUP_TABLES_DIR / "grouped_ablation_sensitivity_by_model_target.csv"
    group_sens_agg.to_csv(group_sens_agg_path, index=False)
    print("Saved:", group_sens_agg_path)

    for metric in ["mean_l1_prob_shift", "mean_max_abs_prob_shift", "pred_changed_rate"]:
        x = np.arange(len(ABLATION_TARGETS))
        width = 0.35

        plt.figure(figsize=(9, 5))

        for i, model in enumerate([BASE_RUN_NAME, FT_RUN_NAME]):
            vals = []
            for target in ABLATION_TARGETS:
                sub = group_sens_agg[
                    (group_sens_agg["model_label"] == model)
                    & (group_sens_agg["target"] == target)
                ]
                vals.append(float(sub[metric].iloc[0]) if len(sub) else np.nan)

            plt.bar(
                x + (i - 0.5) * width,
                vals,
                width,
                label=model,
            )

        plt.xticks(x, ABLATION_TARGETS, rotation=20, ha="right")
        plt.ylabel(metric)
        plt.title(f"Grouped ablation sensitivity: {metric}")
        plt.legend()
        plt.tight_layout()

        savefig_grouped(GROUP_PLOTS_DIR / f"grouped_ablation_sensitivity_{metric}.png")


# 4) Grouped response distribution plots.
group_bar_records = []

for target in ABLATION_TARGETS:
    classes = classes_by_target[target]
    true_col = f"true_{target}"
    pred_col = f"pred_{target}"

    conditions = ["baseline_full_profile"] + [
        c for c in sorted(group_scores["condition"].unique())
        if c != "baseline_full_profile"
    ]

    for condition in conditions:
        sub = group_scores[
            (group_scores["target"] == target)
            & (group_scores["condition"] == condition)
        ].copy()

        if sub.empty:
            continue

        true_sub = sub.drop_duplicates(["ablation_sample_row_id", "target", "condition"])
        true_dist = unweighted_class_distribution(true_sub, true_col, classes)
        true_dist["series"] = "True sampled rows"

        pred_parts = [true_dist]
        prob_parts = []

        for model in [BASE_RUN_NAME, FT_RUN_NAME]:
            m = sub[sub["model_label"] == model].copy()
            if m.empty:
                continue

            pred_dist = unweighted_class_distribution(m, pred_col, classes)
            pred_dist["series"] = f"{model} top pred"
            pred_parts.append(pred_dist)

            prob_dist = mean_probability_distribution(m, classes)
            prob_dist["series"] = f"{model} mean prob"
            prob_parts.append(prob_dist)

        pred_plot = pd.concat(pred_parts, ignore_index=True)
        pred_plot["target"] = target
        pred_plot["condition"] = condition
        pred_plot["plot_type"] = "grouped_top_prediction_distribution"
        group_bar_records.append(pred_plot)

        grouped_bar(
            pred_plot,
            title=f"Grouped ablation true vs top-predicted distribution: {target} / {condition}",
            ylabel="Share of sampled rows",
            path=GROUP_PLOTS_DIR / f"grouped_dist_top_pred_{safe_name(target)}_{safe_name(condition)}.png",
        )

        if prob_parts:
            prob_plot = pd.concat(prob_parts, ignore_index=True)
            prob_plot["target"] = target
            prob_plot["condition"] = condition
            prob_plot["plot_type"] = "grouped_mean_probability_distribution"
            group_bar_records.append(prob_plot)

            grouped_bar(
                prob_plot,
                title=f"Grouped ablation mean predicted probability distribution: {target} / {condition}",
                ylabel="Mean predicted probability",
                path=GROUP_PLOTS_DIR / f"grouped_dist_mean_prob_{safe_name(target)}_{safe_name(condition)}.png",
            )

group_bar_summary = pd.concat(group_bar_records, ignore_index=True) if group_bar_records else pd.DataFrame()
group_bar_summary_path = GROUP_TABLES_DIR / "grouped_distribution_bar_summary.csv"
group_bar_summary.to_csv(group_bar_summary_path, index=False)
print("Saved:", group_bar_summary_path)

display(group_bar_summary.head(30))


# ---------------------------------------------------------------------
# Final grouped manifest
# ---------------------------------------------------------------------
group_manifest = []
for p in sorted(
    list(GROUP_SCORES_DIR.rglob("*"))
    + list(GROUP_TABLES_DIR.rglob("*"))
    + list(GROUP_PLOTS_DIR.rglob("*"))
):
    if p.is_file():
        group_manifest.append({
            "path": str(p),
            "size_mb": p.stat().st_size / 1e6,
        })

group_manifest_df = pd.DataFrame(group_manifest)
group_manifest_path = GROUP_TABLES_DIR / "grouped_output_manifest.csv"
group_manifest_df.to_csv(group_manifest_path, index=False)

print("Saved grouped output manifest:", group_manifest_path)
display(group_manifest_df)